### Real data

In [1]:
import numpy as np
import scipy.io
import torch
from torch_geometric.data import Data
from pathlib import Path
from torch_geometric.datasets import KarateClub, WikipediaNetwork, Planetoid
from prettytable import PrettyTable
from itertools import chain
import networkx as nx
import random
import torch.nn as nn
from sklearn.metrics import adjusted_rand_score
from sklearn.metrics import accuracy_score
import random
from models import GEE, GNN
from time import time
import matplotlib.pyplot as plt
from scipy.stats import norm
from sklearn.model_selection import StratifiedKFold, KFold, train_test_split
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.preprocessing import StandardScaler

def _dedup_edges(edge: torch.Tensor) -> torch.Tensor:
    """
    Converts a directed edge list to an undirected one and removes duplicate edges.
    This is a helper function to ensure graph consistency.

    Args:
        edge (torch.Tensor): A [2, num_edges] tensor representing the edge index.

    Returns:
        torch.Tensor: A processed edge index tensor with no duplicates or self-loops.
    """
    row = torch.minimum(edge[0], edge[1])
    col = torch.maximum(edge[0], edge[1])
    edge_undirected = torch.stack([row, col], dim=0)   

    return torch.unique(edge_undirected, dim=1)

def load_real_data(name: str, root: str = "../datasets"):
    """
    Loads real-world graph datasets from various sources.
    It supports three loading mechanisms:
      1. PyTorch Geometric's online datasets (e.g., "Cora").
      2. Local .mat files (e.g., "your_dataset.mat").
      3. A directory of .npy files for adj, features, and labels.

    Args:
        name (str): The name of the dataset.
        root (str, optional): The root directory for datasets. Defaults to "./datasets".

    Returns:
        torch_geometric.data.Data: A PyG Data object containing graph info.
    """
    # --- 1. Load from PyG's online library ---
    online_map = {
        "KarateClub":        lambda: KarateClub(),
        "Chameleon":         lambda: WikipediaNetwork(root, name="Chameleon"),
        "Cora":              lambda: Planetoid(root, name="Cora"),
        "Citeseer":          lambda: Planetoid(root, name="Citeseer"),
    }
    if name in online_map:
        ds = online_map[name]()       
        data  = ds[0]
        data.edge_index = _dedup_edges(data.edge_index)  
        data.k = ds.num_classes   
        return data

    # --- 2 & 3. Load from local files ---
    root = Path(root)
    mat_file = root / f"{name}.mat"
    npy_dir  = root / name

    if mat_file.exists():                                  # -------- mat --------
        mat  = scipy.io.loadmat(mat_file)
        edge = torch.tensor(mat["Edge"][:, :2].T, dtype=torch.long) - 1
        edge = _dedup_edges(edge)
        if edge.max() == 1:
            edge -= 1

        lbl_key = "Label" if "Label" in mat else "Y"
        y = torch.tensor(mat[lbl_key].ravel(), dtype=torch.long)
        data = Data(x=None, edge_index=edge, y=y)

    elif npy_dir.is_dir():                                 # -------- npy --------
        adj   = np.load(npy_dir / f"{name}_adj.npy")
        feat  = np.load(npy_dir / f"{name}_feat.npy").astype(np.float32)
        label = np.load(npy_dir / f"{name}_label.npy")

        row, col = np.where(adj > 0)
        edge = torch.tensor(np.vstack([row, col]), dtype=torch.long)
        if np.allclose(adj, adj.T):
            edge = _dedup_edges(edge)
        # if edge.min() == 1:
        #     edge -= 1

        data = Data(x=torch.tensor(feat), edge_index=edge,
                    y=torch.tensor(label, dtype=torch.long))
    else:
        raise FileNotFoundError(f"Dataset '{name}' not found in {root}.")

    data.k = len(torch.unique(data.y))
    data.num_nodes = data.y.shape[0]

    if min(data.y) == 1 and max(data.y) == data.k:
        data.y  -= 1
    
    return data

def stratified_split(labels, train_ratio=0.18, val_ratio=0.02, test_ratio=0.80, n_repeats=100, seed=0):
    """Creates repeated stratified train/validation/test splits for labels.

    Args:
        labels: A 1D array or tensor of node labels.
        train_ratio (float): The approximate proportion for the training set.
        val_ratio (float): The approximate proportion for the validation set.
        n_repeats (int): The number of distinct splits to generate.
        seed (int): The random seed for reproducibility.

    Returns:
        list: A list of dictionaries, each holding 'train', 'val', 'test' indices.
    """
    y = labels.cpu().numpy() if torch.is_tensor(labels) else labels
    rng = np.random.default_rng(seed)
    num_classes = np.unique(y).size
    all_splits = []
    for repeat in range(n_repeats):
        train_idx, val_idx, test_idx = [], [], []
        for c in np.unique(y):

            idx = np.where(y == c)[0]
            idx = rng.permutation(idx)
            n = len(idx)
            if n == 3:
                train_idx.append(idx[:2])
                val_idx.append(idx[2])
            if n <= 2:
                raise ValueError(f"Not enough samples")
            else:
                n_train = max(2, int(np.round(train_ratio * n)))
                n_val = max(1, int(np.round(val_ratio * n)))
                n_test = n - n_train - n_val

                train_idx.append(idx[:n_train])
                val_idx.append(idx[n_train:n_train+n_val])
                test_idx.append(idx[n_train+n_val:])

        train_idx = np.concatenate(train_idx)
        val_idx = np.concatenate(val_idx)
        test_idx = np.concatenate(test_idx)
        all_splits.append({
            'train': train_idx,
            'val': val_idx,
            'test': test_idx,
        })
    return all_splits

def write(file_path, content, dataset):
    """
    Appends experiment results for a given run to a tab-separated file.

    Args:
        file_path (str): The path to the output file.
        content (dict): A dictionary mapping method names to their results.
        dataset (str): The dataset we used.
    """
    with open(file_path, "a") as f:
        for method, arr in content.items():
            arr = np.asarray(arr, dtype=float)
            # flat = arr.flatten()
            ari_str = ",".join(f"{x}" for x in arr)
            line = f"{dataset}\t{method}\t{ari_str}\n"
            f.write(line)

def lda_eval(emb, idx, y_true, solver="lsqr", shrinkage="auto"):
    """
    Evaluates node embeddings using a Linear Discriminant Analysis (LDA) classifier.

    Args:
        emb: A 2D array or tensor of node embeddings.
        idx (dict): A dictionary containing 'train', 'val', and 'test' indices.
        y_true (torch.Tensor): The ground-truth labels for all nodes.

    Returns:
        float: The classification accuracy on the test set.
    """
    if isinstance(emb, torch.Tensor):
        emb = emb.detach().cpu().numpy()

    idx_train_all = np.concatenate([idx["train"], idx["val"]])
    idx_test = idx["test"]

    X_train, y_train = emb[idx_train_all], y_true[idx_train_all]
    X_test,  y_test  = emb[idx_test],      y_true[idx_test]

    lda = LinearDiscriminantAnalysis(solver=solver, shrinkage=shrinkage)
    lda.fit(X_train, y_train)
    y_pred = lda.predict(X_test)
    acc    = accuracy_score(y_test, y_pred)
    return acc
   

In [2]:
def run_realdata(data, edge_list, cv_splits, *, gnn_kwargs = None):
    """Orchestrates a comparative experiment of different models on a dataset.

    Args:
        data: The PyTorch Geometric `Data` object for the dataset.
        edge_list (list): The edge list format required by the GEE model.
        cv_splits (list): A list of train/val/test splits, one for each repetition.
        gnn_kwargs (dict, optional): Keyword arguments for the GNN training function.

    Returns:
        tuple: A tuple containing dictionaries for accuracies (`acc_all`),
               execution times (`time_all`), and diagnostic info (`grad_infs`).
    """
    torch.manual_seed(0)
    acc_all = {'GEE':[], 'GNN':[], "GG":[], "GG2":[]}
    time_all = {'GEE':[], 'GNN':[], "GG":[], "GG2":[]}
    loss_all = {}
    grad_all = {}
    stats_acc = {}
    stats_time = {}
    n_reps = len(cv_splits)

    for rep in range(n_reps):
        idx = cv_splits[rep]
        y_true = data.y[idx["test"]]

        # ----------  GEE ----------
        y_train = np.asarray(data.y).copy().reshape(-1,1)
        y_train[idx["val"]] = -1
        y_train[idx["test"]] = -1
        start=time(); Z = GEE.GEE(data.num_nodes, edge_list, y_train); end=time(); t_GEE= end-start
        # y_pred_GEE = torch.argmax(Z, dim=1)[idx["test"]]
        # acc_GEE = accuracy_score(y_true, y_pred_GEE)

        
        # ----------  GNN, GG ----------    
        d_GNN = data.clone()
        d_GG = data.clone()
        d_GG.x = Z
        d_GG2 = data.clone()
        d_GG2.x = Z
        for m in ("train", "val", "test"):
            mask = torch.zeros(d_GNN.num_nodes, dtype=torch.bool)
            mask[idx[m]] = True
            setattr(d_GNN, f"{m}_mask", mask)
            setattr(d_GG, f"{m}_mask", mask)
            setattr(d_GG2, f"{m}_mask", mask)
        
        if rep == 0:
            gnn_kwargs['return_grad'] = True

        start=time(); model_GNN, logits_GNN, grad_inf_GNN = GNN.train(d_GNN, **gnn_kwargs); end=time(); t_GNN= end-start
        y_pred_GNN = logits_GNN.argmax(dim=1)[idx["test"]]
        acc_GNN = accuracy_score(y_true, y_pred_GNN)

        start=time(); model_GG, logits_GG, grad_inf_GG = GNN.train(d_GG, **gnn_kwargs); end=time(); t_GG= end-start
        y_pred_GG = logits_GG.argmax(dim=1)[idx["test"]]
        acc_GG = accuracy_score(y_true, y_pred_GG)

        Z_std  = StandardScaler().fit_transform(Z.detach().cpu().numpy())
        GG_std = StandardScaler().fit_transform(logits_GG.detach().cpu().numpy())

        logits_GG2  = np.concatenate([GG_std, Z_std], axis=1)
        
        acc_GEE = lda_eval(Z,          idx, data.y)
        # acc_GNN = lda_eval(logits_GNN, idx, data.y)
        # acc_GG  = lda_eval(logits_GG,  idx, data.y)
        acc_GG2 = lda_eval(logits_GG2, idx, data.y)

        if gnn_kwargs['return_grad'] == True:
            loss_all['GNN'] = grad_inf_GNN[0]
            loss_all['GG'] = grad_inf_GG[0]
            grad_all['GNN'] = grad_inf_GNN[1]
            grad_all['GG'] = grad_inf_GG[1]
            gnn_kwargs['return_grad'] = False

        acc_list = [acc_GEE, acc_GNN, acc_GG, acc_GG2]
        time_list = [t_GEE, t_GNN, t_GG, t_GG]
        for i, m in enumerate(acc_all):
            acc_all[m].append(acc_list[i])
            time_all[m].append(time_list[i])

            # print(f"  Fold {fold+1}: GEE = {acc_GEE:.4f}, GNN = {acc_GNN:.4f}, GG = {acc_GG:.4f}")

    for m in acc_all:
        arr = np.array(acc_all[m])
        stats_acc[m] = arr.mean(), arr.std(ddof=1)/np.sqrt(100)

        # arr = np.array(time_lists[m])
        # stats_time[m] = arr.mean(), arr.std(ddof=1)/np.sqrt(5)

    print(f"GEE = {stats_acc['GEE'][0]:.4f}±{stats_acc['GEE'][1]:.4f}, \
        GNN = {stats_acc['GNN'][0]:.4f}±{stats_acc['GNN'][1]:.4f},\
            GG = {stats_acc['GG'][0]:.4f}±{stats_acc['GG'][1]:.4f},\
                GG2 = {stats_acc['GG2'][0]:.4f}±{stats_acc['GG2'][1]:.4f}")


    grad_infs = grad_all, loss_all if grad_inf_GNN != None else None
    
    return acc_all, time_all, grad_infs

In [3]:
# --- 1. Define the list of datasets to be analyzed ---
# Datasets are categorized by their source format (.npy, .mat, or integrated).
data_npy_list = ["ACM","BAT","DBLP","EAT","UAT", "Wiki"]
data_mat_list = ["Gene", "IIP", "lastfm", "polblogs", "TerroristRel"]
data_int_list = ["KarateClub", "Chameleon", "Cora", "Citeseer"]
data_list = data_npy_list +  data_mat_list + data_int_list

# --- 2. Iterate through datasets to collect statistics ---
# We will store statistics in a dictionary for easy table generation.
rows = {
    "num_nodes" : [],
    "num_edges" : [],
    "num_classes": [],
    "feat_dim"  : [],
}

for name in data_list:
    data = load_real_data(name) 
    print(data)          
    rows["num_nodes" ].append(data.y.shape[0])
    rows["num_edges" ].append(data.edge_index.size(1))
    rows["num_classes"].append(data.k)
    rows["feat_dim"  ].append(None if data.x is None else data.x.size(1))

# --- 3. Format and print the statistics using PrettyTable ---
table_long = PrettyTable()
table_long.field_names = ["Info"] + data_list
for info, vals in rows.items():
    table_long.add_row([info] + vals)

print(table_long)

Data(x=[3025, 1870], edge_index=[2, 13128], y=[3025], k=3, num_nodes=3025)
Data(x=[131, 81], edge_index=[2, 1074], y=[131], k=4, num_nodes=131)
Data(x=[4057, 334], edge_index=[2, 3528], y=[4057], k=4, num_nodes=4057)
Data(x=[399, 203], edge_index=[2, 5995], y=[399], k=4, num_nodes=399)
Data(x=[1190, 239], edge_index=[2, 13599], y=[1190], k=4, num_nodes=1190)
Data(x=[2405, 4973], edge_index=[2, 16523], y=[2405], k=17, num_nodes=2405)
Data(edge_index=[2, 1672], y=[1103], k=2, num_nodes=1103)
Data(edge_index=[2, 630], y=[219], k=3, num_nodes=219)
Data(edge_index=[2, 27806], y=[7624], k=18, num_nodes=7624)
Data(edge_index=[2, 16715], y=[1224], k=2, num_nodes=1224)
Data(edge_index=[2, 8592], y=[881], k=3, num_nodes=881)
Data(x=[34, 34], edge_index=[2, 78], y=[34], train_mask=[34], k=4)
Data(x=[2277, 2325], edge_index=[2, 31421], y=[2277], train_mask=[2277, 10], val_mask=[2277, 10], test_mask=[2277, 10], k=5)
Data(x=[2708, 1433], edge_index=[2, 5278], y=[2708], train_mask=[2708], val_mask=[2

In [ ]:
# =============================================================================
# Main script for evaluating models on real-world graph datasets.
#
# This script systematically evaluates the performance of different models
# (e.g., GEE, GNN, GG) across various datasets by varying the k-fold
# cross-validation setup (k = 2, 5, 10, 20). Multiple replications are run
# for each setting to ensure robust results.
# =============================================================================
torch.manual_seed(901)
random.seed(901)
np.random.seed(901)
n_reps = 100

# G, Y = generate_sbm(n, k, pq, r)
# G.add_edges_from((i, i) for i in range(n))

for num_folds in [2, 5, 10, 20]:
    percent = int(100 / num_folds)
    for dataname in data_list:
         
        data = load_real_data(dataname)
        edge_list = [(int(data.edge_index[0, i]), int(data.edge_index[1, i]), 1) for i in range(data.edge_index.shape[1])]
        initializer = nn.init.xavier_uniform_
        z = initializer(torch.empty(data.num_nodes, data.k))
        data.x = z
        cv_splits = stratified_split(data.y, train_ratio=0.9/num_folds, val_ratio=0.1/num_folds, test_ratio=1/num_folds, n_repeats=n_reps)

        gnn_kwargs = dict(k=data.k, lr=0.001, num_epochs=10000, patience=100, concat= False, return_grad=False)

        acc_all, time_all, grad_infs = run_realdata(data, edge_list, cv_splits, gnn_kwargs=gnn_kwargs)

        write(f"results/real_data/{percent}%/acc.txt", acc_all, dataname)
        write(f"results/real_data/{percent}%/time.txt", time_all, dataname)

        # if grad_infs != None:
        #     grad_all, loss_all = grad_infs
        
            # write(f"results/DC_SBM/{num_folds}fold/grad.txt", grad_all, r)
            # write(f"results/DC_SBM/{num_folds}fold/loss.txt", loss_all, r)
            # write(f"results/DC_SBM/{num_folds}fold/val.txt", val_all, r)

        



In [4]:
# torch.manual_seed(901)
# random.seed(901)
# np.random.seed(901)
# num_folds = 5
# print(data_list)

# for dataname in data_list:
#     print(dataname)
#     data = load_real_data(dataname)
#     edge_list = [(int(data.edge_index[0, i]), int(data.edge_index[1, i]), 1) for i in range(data.edge_index.shape[1])]
#     initializer = nn.init.xavier_uniform_
#     z = initializer(torch.empty(data.num_nodes, data.k))
#     data.x = z
#     classes, counts = np.unique(data.y, return_counts=True)
#     print(classes, counts)  
#     # all_splits = stratified_split(data.y, train_ratio=0.9/num_folds, val_ratio=0.1/num_folds, test_ratio=1/num_folds, n_repeats=2)



['ACM', 'BAT', 'DBLP', 'EAT', 'UAT', 'Wiki', 'Gene', 'IIP', 'lastfm', 'polblogs', 'TerroristRel', 'KarateClub', 'Chameleon', 'Cora', 'Citeseer']
TerroristRel
[0 1 2] [ 67 181 633]
